# Aktivnost 3 - eksplorativna analiza i priprema podataka

**Tema:** Predikcija cena nekretnina primenom metoda masinskog ucenja  
**Cilj:** ucitati pripremljeni skup `data/nekretnine_raw.csv`, proveriti kvalitet podataka, uraditi EDA, ocistiti duplikate/outlier-e, napraviti izvedene atribute i pripremiti preprocessing tok za modelovanje.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "nekretnine_raw.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_raw_dataset, validate_raw_dataset
from src.data_cleaning import (
    COMPOSITE_PRIMARY_KEY,
    build_cleaned_dataset,
    build_deduped_dataset,
    create_cleaning_summary,
    split_complete_and_incomplete_key_rows,
)
from src.features import (
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    add_model_features,
)
from src.preprocessing import build_preprocessor, get_model_feature_columns
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## Ucitavanje i validacija skupa

Analiza koristi iskljucivo lokalni fajl `data/nekretnine_raw.csv`. Funkcije iz `src.data` proveravaju da skup ima dovoljan broj redova, ciljnu kolonu `price_eur` i ocekivane real-estate atribute.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "nekretnine_raw.csv"  # data/nekretnine_raw.csv
raw_df = load_raw_dataset(DATA_PATH)
validation_summary = validate_raw_dataset(raw_df)

print(f"Broj redova: {raw_df.shape[0]:,}".replace(",", "."))
print(f"Broj kolona: {raw_df.shape[1]}")
display(pd.Series(validation_summary).to_frame("vrednost"))
display(raw_df.head())
display(raw_df.dtypes.rename("dtype").to_frame())

## Pocetna EDA

Pre ciscenja proveravamo nedostajuce vrednosti, tacne duplikate, duplirane URL-ove i osnovnu statistiku numerickih kolona. Ovo daje sliku koliko je skup spreman za modelovanje pre dodatne obrade.


In [ ]:
missing_table = pd.DataFrame(
    {
        "missing_count": raw_df.isna().sum(),
        "missing_pct": (raw_df.isna().mean() * 100).round(2),
    }
).sort_values("missing_count", ascending=False)

numeric_description = (
    raw_df[["area_m2", "price_eur", "rooms", "year_built"]]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .round(2)
)

print("Tacni duplirani redovi:", int(raw_df.duplicated().sum()))
print("Duplirani URL-ovi:", int(raw_df["url"].duplicated().sum()))
display(missing_table)
display(numeric_description)

## Duplikati po kompozitnom kljucu

Tacni duplikati i URL nisu dovoljni za oglase koji imaju isti sadrzaj, ali razlicite URL adrese. Zato koristimo kompozitni kljuc `title + description + area_m2 + price_eur + city`. Redovi sa kompletnim kljucem se deduplikuju, dok se redovi sa nepotpunim kljucem zadrzavaju do provere kvaliteta.


In [ ]:
composite_primary_key = COMPOSITE_PRIMARY_KEY
complete_key_df, incomplete_key_df = split_complete_and_incomplete_key_rows(raw_df)
composite_duplicate_rows_df = complete_key_df[
    complete_key_df.duplicated(subset=composite_primary_key, keep=False)
].copy()
composite_duplicate_rows = len(composite_duplicate_rows_df)
composite_duplicate_groups = composite_duplicate_rows_df.groupby(
    composite_primary_key, dropna=False
).size()
rows_removed_if_keeping_first = int(
    complete_key_df.duplicated(subset=composite_primary_key, keep="first").sum()
)
deduped_df = build_deduped_dataset(raw_df)
duplicate_summary = create_cleaning_summary(raw_df)

display(pd.Series(duplicate_summary).to_frame("vrednost"))
display(
    composite_duplicate_groups.sort_values(ascending=False)
    .head(10)
    .to_frame("broj_ponavljanja")
)

## Ciscenje outlier-a i nevalidnih vrednosti

Za model-ready skup zadrzavamo realne opsege: `area_m2` od 10 do 500, `price_eur` od 10.000 do 1.000.000, `rooms` nedostaje ili je od 0.5 do 10, a `year_built` nedostaje ili je od 1800 do 2028. Pravila su implementirana u `src.data_cleaning.build_cleaned_dataset`.


In [ ]:
valid_area = deduped_df["area_m2"].between(10, 500)
valid_price = deduped_df["price_eur"].between(10_000, 1_000_000)
valid_rooms = deduped_df["rooms"].isna() | deduped_df["rooms"].between(0.5, 10)
valid_year = deduped_df["year_built"].isna() | deduped_df["year_built"].between(
    1800, 2028
)
valid_model_rows = valid_area & valid_price & valid_rooms & valid_year

cleaned_df = build_cleaned_dataset(raw_df)
model_df = add_model_features(cleaned_df)

before_after_summary = pd.DataFrame(
    [
        {"faza": "Pre ciscenja", "redovi": len(raw_df)},
        {"faza": "Posle composite key deduplikacije", "redovi": len(deduped_df)},
        {"faza": "Posle ciscenja", "redovi": len(cleaned_df)},
    ]
)
before_after_summary["uklonjeno_u_odnosu_na_prethodnu_fazu"] = (
    before_after_summary["redovi"]
    .shift(1)
    .sub(before_after_summary["redovi"])
    .fillna(0)
    .astype(int)
)

outlier_summary = pd.DataFrame(
    [
        {
            "pravilo": "area_m2 van opsega 10-500 ili nedostaje",
            "broj_redova": int((~valid_area).sum()),
        },
        {
            "pravilo": "price_eur van opsega 10.000-1.000.000 ili nedostaje",
            "broj_redova": int((~valid_price).sum()),
        },
        {
            "pravilo": "rooms van opsega 0.5-10",
            "broj_redova": int((~valid_rooms).sum()),
        },
        {
            "pravilo": "year_built van opsega 1800-2028",
            "broj_redova": int((~valid_year).sum()),
        },
        {
            "pravilo": "ukupno uklonjeno posle deduplikacije",
            "broj_redova": int((~valid_model_rows).sum()),
        },
    ]
)
outlier_examples = deduped_df.loc[
    ~valid_model_rows,
    ["title", "area_m2", "price_eur", "rooms", "year_built", "city", "url"],
].head(10)

print("Ociscen skup:", cleaned_df.shape)
display(before_after_summary)
display(outlier_summary)
display(outlier_examples)

## Distribucije pre i posle ciscenja

Grafici prikazuju kako deduplikacija i outlier filter menjaju raspodelu cena i kvadrature. Cene su prikazane u hiljadama EUR radi citljivosti.


In [ ]:
price_upper = raw_df["price_eur"].quantile(0.99)
area_upper = raw_df["area_m2"].quantile(0.99)

raw_price_plot = raw_df["price_eur"].dropna().clip(upper=price_upper) / 1000
cleaned_price_plot = cleaned_df["price_eur"].dropna().clip(upper=price_upper) / 1000
raw_area_plot = raw_df["area_m2"].dropna().clip(upper=area_upper)
cleaned_area_plot = cleaned_df["area_m2"].dropna().clip(upper=area_upper)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(raw_price_plot, bins=45, color="tab:blue", ax=axes[0, 0])
axes[0, 0].set_title("Cena pre ciscenja")
axes[0, 0].set_xlabel("Cena (hiljade EUR)")
axes[0, 0].set_ylabel("Broj oglasa")

sns.histplot(cleaned_price_plot, bins=45, color="tab:green", ax=axes[0, 1])
axes[0, 1].set_title("Cena posle ciscenja")
axes[0, 1].set_xlabel("Cena (hiljade EUR)")
axes[0, 1].set_ylabel("Broj oglasa")

sns.histplot(raw_area_plot, bins=45, color="tab:blue", ax=axes[1, 0])
axes[1, 0].set_title("Kvadratura pre ciscenja")
axes[1, 0].set_xlabel("Kvadratura (m2)")
axes[1, 0].set_ylabel("Broj oglasa")

sns.histplot(cleaned_area_plot, bins=45, color="tab:green", ax=axes[1, 1])
axes[1, 1].set_title("Kvadratura posle ciscenja")
axes[1, 1].set_xlabel("Kvadratura (m2)")
axes[1, 1].set_ylabel("Broj oglasa")
plt.tight_layout()

## Kategoricke raspodele i zastupljenost gradova

Uporedjujemo najcesce gradove pre i posle ciscenja, jer je najvise oglasa koncentrisano u nekoliko vecih trzista. Posebno prikazujemo i procenat oglasa po gradu u ociscenom skupu.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
raw_df["city"].value_counts().head(10).plot(kind="bar", ax=axes[0], color="tab:blue")
axes[0].set_title("Top 10 gradova - pre ciscenja")
axes[0].set_xlabel("Grad")
axes[0].set_ylabel("Broj oglasa")
axes[0].tick_params(axis="x", rotation=45)

cleaned_df["city"].value_counts().head(10).plot(
    kind="bar", ax=axes[1], color="tab:green"
)
axes[1].set_title("Top 10 gradova - posle ciscenja")
axes[1].set_xlabel("Grad")
axes[1].set_ylabel("Broj oglasa")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()

city_before_after = (
    pd.DataFrame(
        {
            "Pre ciscenja": raw_df["city"].value_counts(),
            "Posle ciscenja": cleaned_df["city"].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
)
city_before_after["uklonjeno"] = (
    city_before_after["Pre ciscenja"] - city_before_after["Posle ciscenja"]
)
city_percentages = (
    (cleaned_df["city"].value_counts(normalize=True) * 100)
    .round(2)
    .rename("Procenat (%)")
    .to_frame()
)
city_percentages["Broj oglasa"] = cleaned_df["city"].value_counts()

display(city_before_after.sort_values("Pre ciscenja", ascending=False).head(10))
display(city_percentages.head(15))

## Cena po kvadratu i regionalni obrasci

Pored ukupnih raspodela posmatramo cenu po m2 za najcesce gradove i percentile cena za vodece regione u Beogradu i Novom Sadu. Ove tabele i grafikoni pokazuju da lokacija ima jak uticaj na cenu.


In [ ]:
top_cleaned_cities = cleaned_df["city"].value_counts().head(8).index
price_per_m2_by_city = model_df[model_df["city"].isin(top_cleaned_cities)].copy()
price_per_m2_by_city["price_per_m2_plot"] = price_per_m2_by_city["price_per_m2"].clip(
    upper=price_per_m2_by_city["price_per_m2"].quantile(0.98)
)

plt.figure(figsize=(12, 5))
sns.boxplot(data=price_per_m2_by_city, x="city", y="price_per_m2_plot")
plt.title("Cena po m2 za najcesce gradove posle ciscenja")
plt.xlabel("Grad")
plt.ylabel("Cena po m2 (EUR, ograniceno na 98. percentil)")
plt.xticks(rotation=35)
plt.tight_layout()

region_price_summary = (
    cleaned_df[cleaned_df["city"].isin(["Beograd", "Novi Sad"])]
    .groupby(["city", "region"])["price_eur"]
    .agg(
        broj_oglasa="count",
        p05=lambda s: s.quantile(0.05),
        p25=lambda s: s.quantile(0.25),
        medijan="median",
        p75=lambda s: s.quantile(0.75),
        p95=lambda s: s.quantile(0.95),
    )
    .query("broj_oglasa >= 30")
    .sort_values(["city", "medijan"], ascending=[True, False])
    .round(0)
)
display(region_price_summary.groupby(level=0).head(10))

## Korelacije

Raw Pearson korelacija cene i kvadrature je osetljiva na ekstremne vrednosti. Zato poredimo raw i cleaned korelacije i prikazujemo korelacionu matricu na `model_df`.


In [ ]:
raw_correlation_df = raw_df[["area_m2", "price_eur"]].dropna()
cleaned_correlation_df = cleaned_df[["area_m2", "price_eur"]].dropna()
correlation_comparison = pd.DataFrame(
    [
        {
            "metrika": "Pearson raw_df",
            "area_price_correlation": raw_correlation_df["area_m2"].corr(
                raw_correlation_df["price_eur"], method="pearson"
            ),
        },
        {
            "metrika": "Spearman raw_df",
            "area_price_correlation": raw_correlation_df["area_m2"].corr(
                raw_correlation_df["price_eur"], method="spearman"
            ),
        },
        {
            "metrika": "Pearson cleaned_df",
            "area_price_correlation": cleaned_correlation_df["area_m2"].corr(
                cleaned_correlation_df["price_eur"], method="pearson"
            ),
        },
        {
            "metrika": "Spearman cleaned_df",
            "area_price_correlation": cleaned_correlation_df["area_m2"].corr(
                cleaned_correlation_df["price_eur"], method="spearman"
            ),
        },
    ]
).set_index("metrika")
display(correlation_comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(
    data=raw_df.sample(min(3000, len(raw_df)), random_state=42),
    x="area_m2",
    y="price_eur",
    alpha=0.25,
    ax=axes[0],
)
axes[0].set_title("Cena i kvadratura pre ciscenja")
axes[0].set_xlim(0, raw_df["area_m2"].quantile(0.99))
axes[0].set_ylim(0, raw_df["price_eur"].quantile(0.99))

sns.scatterplot(
    data=cleaned_df.sample(min(3000, len(cleaned_df)), random_state=42),
    x="area_m2",
    y="price_eur",
    alpha=0.25,
    ax=axes[1],
    color="tab:green",
)
axes[1].set_title("Cena i kvadratura posle ciscenja")
axes[1].set_xlim(0, cleaned_df["area_m2"].quantile(0.99))
axes[1].set_ylim(0, cleaned_df["price_eur"].quantile(0.99))
plt.tight_layout()

correlation_columns = [
    "price_eur",
    "area_m2",
    "rooms",
    "year_built",
    "floor",
    "total_floors",
    "building_age",
    "price_per_m2",
]
plt.figure(figsize=(9, 6))
sns.heatmap(
    model_df[correlation_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
)
plt.title("Korelaciona matrica numerickih atributa")
plt.tight_layout()

## Feature engineering

Funkcija `src.features.add_model_features` dodaje domenske atribute koji ce se koristiti u modelima: `price_per_m2`, parsiranu spratnost, indikator poslednjeg sprata, starost zgrade i tekstualne indikatore za luksuzne/penthouse/dupleks oglase. Ciljna kolona `price_eur` se koristi samo kao `y`, ne kao ulazni atribut.


In [ ]:
engineered_columns = [
    "price_per_m2",
    "floor",
    "total_floors",
    "is_last_floor",
    "building_age",
    "is_lux",
    "is_penthouse",
    "is_duplex",
]
feature_contract = pd.DataFrame(
    {
        "numeric_features": pd.Series(MODEL_NUMERIC_FEATURES),
        "categorical_features": pd.Series(MODEL_CATEGORICAL_FEATURES),
    }
)

print("Izvedene kolone:", engineered_columns)
print(
    "price_eur nije u ulaznim kolonama:", "price_eur" not in get_model_feature_columns()
)
display(model_df[engineered_columns + ["price_eur", "city", "region"]].head())
display(feature_contract)

## Preprocessing za modelovanje

Na `model_df` primenjujemo zajednicki preprocessing iz `src.preprocessing.build_preprocessor`. Interno se koristi `ColumnTransformer` sa `SimpleImputer`, `StandardScaler` i `OneHotEncoder`: numericke vrednosti se imputiraju medianom i skaliraju, a kategorije se imputiraju vrednoscu `Nepoznato` i one-hot enkodiraju. `train_test_split` se radi pre fitovanja da ne bi doslo do curenja informacija iz test skupa.


In [ ]:
target = "price_eur"
feature_columns = get_model_feature_columns()
X = model_df[feature_columns]
y = model_df[target]

preprocessor = build_preprocessor()  # ColumnTransformer + SimpleImputer + OneHotEncoder
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)


def transformed_missing_count(matrix):
    array = matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)
    return int(np.isnan(array).sum())


preprocessing_summary = pd.DataFrame(
    [
        {
            "skup": "X_train",
            "redovi": X_train.shape[0],
            "kolone_pre": X_train.shape[1],
            "kolone_posle": X_train_prepared.shape[1],
            "missing_posle": transformed_missing_count(X_train_prepared),
        },
        {
            "skup": "X_test",
            "redovi": X_test.shape[0],
            "kolone_pre": X_test.shape[1],
            "kolone_posle": X_test_prepared.shape[1],
            "missing_posle": transformed_missing_count(X_test_prepared),
        },
    ]
)

print("Pre ciscenja:", raw_df.shape)
print("Posle ciscenja:", model_df.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
print(
    "Nedostajuce vrednosti posle preprocessinga:",
    int(preprocessing_summary["missing_posle"].sum()),
)
display(preprocessing_summary)
display(model_df[feature_columns + [target, "price_per_m2"]].head())

## Zakljucak aktivnosti 3

Aktivnost 3 je pripremila podatke za narednu fazu: skup se ucitava offline, validira se schema, rade se EDA i vizualizacije, uklanjaju se duplikati i ocigledni outlier-i, kreiraju se domenski atributi i dobija se preprocessing tok koji se moze direktno koristiti u modelovanju. Svi koraci polaze od `data/nekretnine_raw.csv` i ne zavise od ponovnog pokretanja scraper-a.
